# 01 Bronze ingest

Raw volume JSON → **schema-enforced** Delta tables. Bronze layer keeps the data as parsed
(no cleaning); it only enforces a schema and makes re-ingestion idempotent.

- **`bronze_battles`** — one row per `battle_id`, loaded with an idempotent `MERGE`.
  The same battle appears in two players' logs, so the `MERGE` collapses those to one row.
- **`bronze_cards`** — small, slowly-changing dim; full-refresh overwrite each run.
- **`bronze_players`** — the crawl seed, kept for lineage (overwrite).

## Config

In [0]:
from pyspark.sql import functions as F, Window, types as T
from delta.tables import DeltaTable

CATALOG, SCHEMA = "workspace", "clash"
RAW = f"/Volumes/{CATALOG}/{SCHEMA}/raw"

BRONZE_BATTLES = f"{CATALOG}.{SCHEMA}.bronze_battles"
BRONZE_CARDS   = f"{CATALOG}.{SCHEMA}.bronze_cards"
BRONZE_PLAYERS = f"{CATALOG}.{SCHEMA}.bronze_players"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

DataFrame[]

## Bronze battles
Schema enforcement, deduplication, and idempotent merge are applied in the bronze battlelog layer.

In [0]:
battles_schema = T.StructType([
    T.StructField("battle_id", T.StringType()),
    T.StructField("battle_time", T.StringType()),
    T.StructField("battle_time_raw", T.StringType()),
    T.StructField("type", T.StringType()),
    T.StructField("is_ladder", T.BooleanType()),
    T.StructField("game_mode", T.StringType()),
    T.StructField("arena_id", T.LongType()),
    T.StructField("arena_name", T.StringType()),
    T.StructField("player_tag", T.StringType()),
    T.StructField("player_name", T.StringType()),
    T.StructField("player_starting_trophies", T.LongType()),
    T.StructField("player_trophy_change", T.LongType()),
    T.StructField("player_crowns", T.LongType()),
    T.StructField("player_cards", T.ArrayType(T.LongType())),
    T.StructField("player_deck_hash", T.StringType()),
    T.StructField("opponent_tag", T.StringType()),
    T.StructField("opponent_name", T.StringType()),
    T.StructField("opponent_starting_trophies", T.LongType()),
    T.StructField("opponent_trophy_change", T.LongType()),
    T.StructField("opponent_crowns", T.LongType()),
    T.StructField("opponent_cards", T.ArrayType(T.LongType())),
    T.StructField("opponent_deck_hash", T.StringType()),
    T.StructField("win", T.BooleanType()),
    T.StructField("source_player_tag", T.StringType()),
    T.StructField("ingested_at", T.StringType()),
])

raw = spark.read.schema(battles_schema).json(f"{RAW}/*/battles.jsonl")

In [0]:
# Deduplicate within this load so each battle_id appears once before the MERGE.
# Always keep the row with the lowest source_player_tag for a battle, so the kept row is deterministic across reruns.
w = Window.partitionBy("battle_id").orderBy("source_player_tag")
staged = (raw
    .withColumn("rn", F.row_number().over(w))
    .filter("rn = 1").drop("rn")
    .withColumn("bronze_loaded_at", F.current_timestamp()))

print(raw.count(), "raw rows ->", staged.count(), "unique battle_id")

319879 raw rows -> 318176 unique battle_id


### Idempotent MERGE

First run creates the Delta table; later runs `MERGE` on `battle_id` and insert only battles not already there. This ensures re-running the pipeline does not create duplicates or overwrite existing battles.

In [0]:
if not spark.catalog.tableExists(BRONZE_BATTLES):
    staged.write.format("delta").saveAsTable(BRONZE_BATTLES)
    print("created", BRONZE_BATTLES)
else:
    (DeltaTable.forName(spark, BRONZE_BATTLES).alias("t")
        .merge(
            source=staged.alias("s"),
            condition="t.battle_id = s.battle_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())
    print("merged into", BRONZE_BATTLES)

created workspace.clash.bronze_battles


In [0]:
bb = spark.table(BRONZE_BATTLES)
print(bb.count(), "rows;", bb.select("battle_id").distinct().count(), "distinct battle_id")
display(bb.limit(10))

318176 rows; 318176 distinct battle_id


battle_id,battle_time,battle_time_raw,type,is_ladder,game_mode,arena_id,arena_name,player_tag,player_name,player_starting_trophies,player_trophy_change,player_crowns,player_cards,player_deck_hash,opponent_tag,opponent_name,opponent_starting_trophies,opponent_trophy_change,opponent_crowns,opponent_cards,opponent_deck_hash,win,source_player_tag,ingested_at,bronze_loaded_at
000006d72b55daa0f7529d2c0354254960699191,2026-06-06T15:08:17+00:00,20260606T150817.000Z,PvP,true,Ladder,54000144,Spirit Square,#808V0UCPY,Nariko15633,13500,30,2,"List(26000007, 26000000, 28000004, 28000008, 28000000, 26000021, 26000042, 27000006)",458e0293c451aaa6c85e42131d38c2eb46c9a791,#VC2U2LQ2V,FreakyOrihime,13500,null,0,"List(26000017, 26000004, 26000006, 28000002, 26000087, 26000007, 28000006, 26000055)",7313a7b6d731380ad7473ea1313779b119aa023a,true,#808V0UCPY,2026-06-08T03:12:38.362826+00:00,2026-06-10T10:23:27.354Z
00002b0ce7136e5880e5b13499f21c619a8521f1,2026-06-07T07:41:23+00:00,20260607T074123.000Z,trail,false,TeamVsTeam,54000152,Legendary Arena,#9C2LYLYC,Jordan,null,null,0,"List(26000056, 26000002, 26000050, 26000053, 28000011, 26000097, 26000040, 28000000)",694905d7fbe104b3ee2935fd08b671606a66f2bd,#J2YGUR8JU,#1 CHARLY,null,null,3,"List(26000055, 26000000, 26000011, 28000005, 26000040, 26000021, 26000030, 28000000)",963e2773e33863b4660a20f4afcccff136e689a6,false,#9C2LYLYC,2026-06-08T03:53:19.583679+00:00,2026-06-10T10:23:27.354Z
0000fe660ba600ed82ccb77230dd283cc7203fd1,2026-06-07T11:56:44+00:00,20260607T115644.000Z,pathOfLegend,false,Ranked1v1_NewArena,54000152,Legendary Arena,#92L20CC2,OT⚡❄️SILVER❄️⚡,null,30,1,"List(26000037, 26000034, 26000063, 28000012, 26000067, 26000068, 28000006, 28000005)",d96ade20fe7d91948b8a251a124c7dba5cabb6f5,#20QQ2GRGL,SamuelH,null,null,0,"List(26000064, 26000000, 26000030, 28000015, 28000018, 26000021, 27000006, 28000014)",0821dcc376f58ab30a993ba779a8d33e588c593f,true,#92L20CC2,2026-06-08T03:33:02.544620+00:00,2026-06-10T10:23:27.354Z
0001b038c2e5b186d424c174208438ddd9eddcf1,2026-06-08T02:51:14+00:00,20260608T025114.000Z,pathOfLegend,false,Ranked1v1_NewArena,54000152,Legendary Arena,#89QVP2G,LORDspartan™,null,30,2,"List(26000022, 26000000, 26000008, 26000012, 26000023, 26000058, 26000032, 28000001)",03093e0713a8755ae43111748f0a33ebfa986775,#8QVJVP09,allBlock,null,null,1,"List(26000055, 26000018, 26000012, 26000021, 28000012, 26000064, 28000011, 26000014)",a2fca2101c6fbf9b0aa156095afb0f256799c004,true,#89QVP2G,2026-06-08T03:05:24.841406+00:00,2026-06-10T10:23:27.354Z
0001f6cccad0e9482c7f3988a52b6e8ce0291e9f,2026-06-06T02:19:03+00:00,20260606T021903.000Z,trail,false,TripleElixir_Friendly,54000152,Legendary Arena,#2GLUUVYY0,Helpzv,5,null,1,"List(26000045, 26000103, 28000017, 26000046, 26000050, 26000025, 28000007, 26000021)",9849c76134c7151798017247972c65a32bc64e7b,#908Q2VUY,Rollin_Uhpp,5,1,2,"List(26000056, 27000001, 27000010, 26000054, 26000020, 28000013, 28000012, 26000080)",b4a09230a372e9826c1d20f44272e1f744858b04,false,#2GLUUVYY0,2026-06-08T03:17:11.025146+00:00,2026-06-10T10:23:27.354Z
00020d11004999328ba18140096f8efdfe49bc97,2026-06-07T23:24:46+00:00,20260607T232446.000Z,PvP,true,Ladder,54000144,Spirit Square,#ULJ2JGGP,yunus,13594,31,1,"List(26000004, 26000074, 26000059, 28000011, 26000027, 28000001, 26000042, 26000005)",77fd7957ef54bdf020212ca657952a14d05d01dd,#RPUJ8UPJ9,MattyDe_oro!!,13607,-31,0,"List(27000013, 26000000, 26000044, 28000017, 26000064, 28000011, 26000041, 26000030)",6eb6ce7c7b07aef3401069d569d05f36dbbc0db1,true,#ULJ2JGGP,2026-06-08T03:42:37.027149+00:00,2026-06-10T10:23:27.354Z
000215fbbc4baffadba28de50414f2d2b79d8d42,2026-06-06T02:57:48+00:00,20260606T025748.000Z,trail,true,Ladder,168000147,Seasonal Arena I,#P8099RLJ,SOSA,14211,30,2,"List(26000011, 26000017, 26000012, 28000008, 26000043, 26000016, 28000006, 28000001)",20794aad1fed4bddfbf1157afc9d3bfa59564975,#9J0PGJ2,Shamah,14211,-30,0,"List(26000047, 26000038, 26000011, 26000021, 28000017, 26000080, 28000007, 26000001)",cabce2ffcf7af2cc7f407ea8f4

## Bronze cards

Loads card dimension data into a schema-enforced Delta table (full overwrite).

In [0]:
cards_schema = T.StructType([
    T.StructField("card_id", T.LongType()),
    T.StructField("name", T.StringType()),
    T.StructField("rarity", T.StringType()),
    T.StructField("elixir_cost", T.LongType()),
    T.StructField("max_level", T.LongType()),
])
cards = (spark.read.schema(cards_schema).option("multiline", "true")
    .json(f"{RAW}/cards.json")
    .withColumn("bronze_loaded_at", F.current_timestamp()))
cards.write.format("delta").mode("overwrite").saveAsTable(BRONZE_CARDS)
print(spark.table(BRONZE_CARDS).count(), "cards")
display(spark.table(BRONZE_CARDS).orderBy("card_id").limit(10))

121 cards


card_id,name,rarity,elixir_cost,max_level,bronze_loaded_at
26000000,Knight,common,3,16,2026-06-10T10:23:41.691Z
26000001,Archers,common,3,16,2026-06-10T10:23:41.691Z
26000002,Goblins,common,2,16,2026-06-10T10:23:41.691Z
26000003,Giant,rare,5,14,2026-06-10T10:23:41.691Z
26000004,P.E.K.K.A,epic,7,11,2026-06-10T10:23:41.691Z
26000005,Minions,common,3,16,2026-06-10T10:23:41.691Z
26000006,Balloon,epic,5,11,2026-06-10T10:23:41.691Z
26000007,Witch,epic,5,11,2026-06-10T10:23:41.691Z
26000008,Barbarians,common,5,16,2026-06-10T10:23:41.691Z
26000009,Golem,epic,8,11,2026-06-10T10:23:41.691Z


## Bronze players (seed lineage)

Not an analytical table, just a record of the crawl seed for player battlelogs (overwrite)

In [0]:
players_schema = T.StructType([
    T.StructField("tag", T.StringType()),
    T.StructField("name", T.StringType()),
    T.StructField("trophies", T.LongType()),
    T.StructField("clan_tag", T.StringType()),
])
players = (spark.read.schema(players_schema).option("multiline", "true")
    .json(f"{RAW}/players.json")
    .withColumn("bronze_loaded_at", F.current_timestamp()))
players.write.format("delta").mode("overwrite").saveAsTable(BRONZE_PLAYERS)
print(spark.table(BRONZE_PLAYERS).count(), "players")
display(spark.table(BRONZE_PLAYERS).limit(10))

9976 players


tag,name,trophies,clan_tag,bronze_loaded_at
#PPVL99U0R,flazrael-antifa,14000,#QRCCY0LJ,2026-06-10T10:23:46.559Z
#V9QC0UYVG,gui,14000,#QRCCY0LJ,2026-06-10T10:23:46.559Z
#GYGLCC28,dtrem7複|,14000,#QRCCY0LJ,2026-06-10T10:23:46.559Z
#9J9UQYJ82,Łʑ☮ɴąɴɗѳ,14000,#QRCCY0LJ,2026-06-10T10:23:46.559Z
#QJ9J2YQ2,Robert,14000,#QRCCY0LJ,2026-06-10T10:23:46.559Z
#8PU88JV0G,LJira¥a,14000,#QRCCY0LJ,2026-06-10T10:23:46.559Z
#QQ2UGVV0L,TEF丨Jerry ✨ CR,14000,#QRCCY0LJ,2026-06-10T10:23:46.559Z
#PCQCJRYYR,Jhonata Br,14000,#QRCCY0LJ,2026-06-10T10:23:46.559Z
#90QLQUG09,Dodowq,14000,#QRCCY0LJ,2026-06-10T10:23:46.559Z
#L9GQJ9PUY,#1 ✨ Storm │優速⚡,14000,#QRCCY0LJ,2026-06-10T10:23:46.559Z


## Done

Bronze is loaded and idempotent. Next: `02_silver_transform`.